In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
import joblib
import warnings
import os
warnings.filterwarnings("ignore")

os.chdir(r"E:\retailpulse")

rfm = pd.read_csv("data/processed/rfm.csv")

# ── Churn Risk based on RFM Segments ──────────────────────
# This is the industry standard approach for marketplace data
# where almost all customers are one-time buyers

def churn_risk(row):
    if row["segment"] in ["Champions","Loyal Customers",
                           "Potential Loyalists"]:
        return "Low Risk"
    elif row["segment"] in ["At Risk","Big Spenders"]:
        return "Medium Risk"
    else:  # Lost, Cant Lose Them, Hibernating
        return "High Risk"

rfm["churn_risk"] = rfm.apply(churn_risk, axis=1)

# Churn probability based on recency (normalized 0-1)
rfm["churn_probability"] = (rfm["recency"] /
                             rfm["recency"].max()).round(4)

# Summary
print("="*50)
print("  CHURN RISK ASSESSMENT")
print("="*50)
risk_summary = rfm.groupby("churn_risk").agg(
    customers     = ("customer_id_unique","count"),
    avg_monetary  = ("monetary","mean"),
    total_revenue = ("monetary","sum")
).round(2).reset_index()

risk_summary["pct"] = (risk_summary["customers"] /
                        len(rfm) * 100).round(1)

print(risk_summary.to_string(index=False))

total_at_risk = rfm[rfm["churn_risk"].isin(
    ["High Risk","Medium Risk"])]["monetary"].sum()
print(f"\n💰 Revenue at risk from churning customers:")
print(f"   BRL {total_at_risk:,.0f}")

# Top high risk customers by spend
high_risk = (rfm[rfm["churn_risk"]=="High Risk"]
             .sort_values("monetary", ascending=False)
             .head(10))

print(f"\n🔴 Top 10 High-Risk customers by spend:")
print(high_risk[["customer_id_unique","segment",
                  "recency","monetary",
                  "churn_probability"]].to_string(index=False))

# Visualize
fig = px.bar(risk_summary,
             x="churn_risk",
             y="customers",
             color="churn_risk",
             title="Customer Churn Risk Distribution",
             text="pct",
             color_discrete_map={
                 "Low Risk":    "#22c55e",
                 "Medium Risk": "#f59e0b",
                 "High Risk":   "#ef4444"},
             labels={"customers":"Number of Customers",
                     "churn_risk":"Risk Level"})
fig.update_traces(texttemplate="%{text}%")
fig.show()

fig2 = px.scatter(rfm.sample(3000, random_state=42),
                  x="recency",
                  y="monetary",
                  color="churn_risk",
                  color_discrete_map={
                      "Low Risk":    "#22c55e",
                      "Medium Risk": "#f59e0b",
                      "High Risk":   "#ef4444"},
                  title="Churn Risk — Recency vs Spend",
                  labels={
                      "recency":"Days Since Last Purchase",
                      "monetary":"Total Spend (BRL)"},
                  opacity=0.6)
fig2.show()

# Save
rfm.to_csv("data/processed/rfm_churn.csv", index=False)
print("\n✅ Churn risk saved!")

  CHURN RISK ASSESSMENT
 churn_risk  customers  avg_monetary  total_revenue  pct
  High Risk      23419        138.92     3253400.83 25.1
   Low Risk      46841        166.70     7808169.46 50.2
Medium Risk      23090        188.69     4356824.54 24.7

💰 Revenue at risk from churning customers:
   BRL 7,610,225

🔴 Top 10 High-Risk customers by spend:
              customer_id_unique        segment  recency  monetary  churn_probability
da122df9eeddfedc1dc1f5349a1a690c Cant Lose Them      515   7571.63             0.7213
dc4802a71eae9be1dd28f5d788ceb526 Cant Lose Them      563   6929.31             0.7885
ff4159b92c40ebe40454e3e6a7c35ed6 Cant Lose Them      462   6726.66             0.6471
eebb5dda148d3893cdaf5b5ca3040ccb Cant Lose Them      498   4764.34             0.6975
edf81e1f3070b9dac83ec83dacdbb9bc Cant Lose Them      498   4194.76             0.6975
5e713be0853d8986528d7869a0811d2b Cant Lose Them      571   4042.74             0.7997
5d09b0d82126457e2a8ebfb9c9a1ffc4 Cant Lose Th


✅ Churn risk saved!
